In [ ]:
import jax
import jax.numpy as jnp
import flax.nnx as nnx
import netket as nk
import netket.experimental as nkx
import sys
sys.path.append('..')
from NES_VMC import NESTotalAnsatz, create_machine,init_sampler_state,\
    generate_random_initial_states,ha,SingleStateAnsatz,create_single_machine,\
        create_machine_matrix,Ham_psi,Ham_Psi,NES_loss_energy,nes_vmc_gradient,hi,E_fcis,mcmc_sampler_multichain,\
        NESFermionHopRule,compute_qgt,sampler_info
import optax
from typing import Callable
from functools import partial
from jax.flatten_util import ravel_pytree
import time

# ========== 你原有全局参数（直接复用） ==========
# 单系统希尔伯特空间
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=2,
    s=1/2,
    n_fermions_per_spin=(1,1),
)
K = 2  # NES 扩展副本数
hi_ext = hi ** K  # 扩展希尔伯特空间
SINGLE_SIZE = hi.size  # 单个子系统维度 = 4
single_edges = ((0, 1), (2, 3))  # 费米子跃迁边
g = nk.graph.Graph(edges=single_edges)
single_rule = nk.sampler.rules.FermionHopRule(hi, graph=g)
tensor_rule = nk.sampler.rules.TensorRule(hi_ext, [single_rule] * K)

total_ansatz = NESTotalAnsatz(4,K,12,rngs=nnx.Rngs(11))
total_machine, total_graphdef,total_params = create_machine(total_ansatz)
total_matrix_machine, total_graphdef,total_params = create_machine_matrix(total_ansatz)

single_machine_list = []
for ansatz in total_ansatz.single_ansatz_list:
    m, g, p = create_single_machine(ansatz)
    single_machine_list.append(m)

/opt/miniconda3/envs/Netket/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


∣NK⟩ Tip: You must cite NetKet according to our policy. Use nk.cite() to find out how.

H₂ FCI 基准能量
E0 = -1.01546825 Ha  |  激发能：0.0000 eV
E1 = -0.87542794 Ha  |  激发能：3.8107 eV
E2 = -0.42938376 Ha  |  激发能：15.9482 eV
E3 = -0.26922131 Ha  |  激发能：20.3064 eV


In [2]:
N_CHAINS = 16
N_WARMUP = 100
N_SAMPLES_PER_CHAIN = 200
SWEEP_SIZE = 30
N_ITER =200
SINGLE_SIZE = hi.size  # 单个子系统维度 = 4

logger.info("训练参数配置:")
logger.info(f"  - N_CHAINS: {N_CHAINS}")
logger.info(f"  - N_WARMUP: {N_WARMUP}")
logger.info(f"  - N_SAMPLES_PER_CHAIN: {N_SAMPLES_PER_CHAIN}")
logger.info(f"  - SWEEP_SIZE: {SWEEP_SIZE}")
logger.info(f"  - N_ITER: {N_ITER}")

total_ansatz = NESTotalAnsatz(4,K,12,rngs=nnx.Rngs(11))
total_machine, total_graphdef,total_params = create_machine(total_ansatz)
total_matrix_machine, total_graphdef,total_params = create_machine_matrix(total_ansatz)

single_machine_list = []
for ansatz in total_ansatz.single_ansatz_list:
    m, g, p = create_single_machine(ansatz)
    single_machine_list.append(m)
    
    

optimizer = optax.sgd(learning_rate=0.01)
opt_state = optimizer.init(total_params)

ext_edges = []
for k in range(K):
    offset = k * SINGLE_SIZE
    for (i, j) in single_edges:
        ext_edges.append((i + offset, j + offset))
ext_edges = jnp.array(ext_edges)  # 转为jax数组（关键修复）

nes_rule = NESFermionHopRule(edges=ext_edges)
nes_sampler = nk.sampler.MetropolisSampler(
    hilbert=hi_ext,
    rule=nes_rule,
    n_chains=16,
    sweep_size=20
)

# 采样器状态初始化（替代原 init_sampler_state）
sampler_rng = jax.random.PRNGKey(21)
sampler_state = nes_sampler.init_state(total_machine, total_params, sampler_rng)

# ==================== 训练循环（仅替换采样部分） ====================
logger.info("\n" + "="*60)
logger.info("开始多链 NES-VMC 训练 (NetKet 自定义采样器 + 朴素梯度下降)")
logger.info("="*60)
logger.info(f"基态能量={E_fcis[0]:.8f} Ha| 第一激发态能量={E_fcis[1]:.8f} Ha| 第二激发态能量={E_fcis[2]:.8f} Ha")

history = {
    'step': [],
    'energy_0st': [],
    'energy_1st': [],
    'energy_std': [],
    'loss': [],
    'params': [],
    'E_Lmatrix':[],
    'natural_grad':[],
    'grad_flat':[],
    'samples':[],
    'log_Psi':[],
    'log_M':[],
    'log_Psi_mean':[],
    'log_Psi_min':[],
    'log_Psi_max':[],
    'grad_norm':[],
}

start_time = time.time()
for step in range(N_ITER):
    # 2. 正式采样
    samples_raw, sampler_state = nes_sampler.sample(
        machine=total_machine, parameters=total_params, 
        state=sampler_state, chain_length=N_SAMPLES_PER_CHAIN
    )
        # 3. 维度重塑，适配梯度函数输入
    samples = samples_raw.reshape(-1, hi_ext.size)
    x_batch = samples.reshape(-1, K, 4)
    # 3. 计算能量和自然梯度（逻辑和原代码一致）
    grad, loss_mean, E_L_mean = nes_vmc_gradient(ha=ha,
                                                 total_matrix_machine=total_matrix_machine,
                                                 total_machine=total_machine,
                                                 single_machine_list=single_machine_list,
                                                 total_params=total_params,
                                                 x_batch=samples.reshape(-1,K,4))
    #grad = jax.tree_util.tree_map(lambda x: x * 2, grad)
    
    grad_flat , grad_unravel_fn = ravel_pytree(grad)
    # qgt_reg, unravel_fn = compute_qgt(total_machine, total_params, samples.reshape(-1,2,4), diag_shift=0.1)
    
    # # # 自然梯度求解
    # natural_grad_flat = jnp.linalg.solve(qgt_reg, grad_flat)
    # natural_grad = grad_unravel_fn(natural_grad_flat)
    # grad = natural_grad
        
    # 4. 更新参数
    updates, opt_state = optimizer.update(grad, opt_state, total_params)
    total_params = optax.apply_updates(total_params, updates)
    
    
    
    log_Psi_batch = total_machine(total_params, samples.reshape(-1,K,4))
    eig_vals, eig_vecs = jnp.linalg.eigh(E_L_mean)
    grad_norm = jnp.linalg.norm(grad_flat)
    
    
    history['step'].append(step)
    history['E_Lmatrix'].append(E_L_mean)
    history['samples'].append(samples)
    history['loss'].append(loss_mean)
    history['log_Psi_mean'].append(log_Psi_batch.mean())
    history['log_Psi_min'].append(log_Psi_batch.min())
    history['log_Psi_max'].append(log_Psi_batch.max())
    history['grad_norm'].append(grad_norm)
    history['energy_0st'].append(eig_vals[0])
    history['energy_1st'].append(eig_vals[1])
    history['params'].append(total_params)
    # 5. 记录历史
    if step % 50 == 0 or step == N_ITER - 1:
        # --------------------- 【NES-VMC 监控模板】直接用 ---------------------
        # 1. 监控 log_Psi
        #log_Psi_batch = total_machine(total_params, samples.reshape(-1,K,4))
        log_msg = f"log_Psi: mean={log_Psi_batch.mean():.3f} | min={log_Psi_batch.min():.3f} | max={log_Psi_batch.max():.3f}"
        logger.info(log_msg)

        # 2. 监控梯度范数
        
        logger.info(f"grad norm = {grad_norm:.4f}")

        # 5. 局域能量矩阵
        #logger.info(f"E_L mean =\n{E_L_mean}")
    
        #eig_vals, eig_vecs = jnp.linalg.eigh(E_L_mean)
        # #history['natural_grad'].append(natural_grad)
        # history['grad_flat'].append(grad_flat)
        # history['log_Psi'].append(log_Psi)
        # history['log_M'].append(log_M)
        
        logger.info(f"Step {step:3d} | Loss: {loss_mean}|0st能量={eig_vals[0]:.8f} Ha| 1st能量={eig_vals[1]:.8f} Ha")
        # logger.info(f'grad={grad_flat[30:31]}')
        logger.info('#-----------------------------------------#')


end_time = time.time()
elapsed_time = end_time - start_time
logger.info(f"训练耗时：{elapsed_time:.2f} 秒")

# 保存历史记录到 JSON 文件
history_to_save = {
    'step': history['step'],
    'energy_0st': [float(x) for x in history['energy_0st']],
    'energy_1st': [float(x) for x in history['energy_1st']],
    'loss': [float(x) for x in history['loss']],
    'log_Psi_mean': [float(x) for x in history['log_Psi_mean']],
    'log_Psi_min': [float(x) for x in history['log_Psi_min']],
    'log_Psi_max': [float(x) for x in history['log_Psi_max']],
    'grad_norm': [float(x) for x in history['grad_norm']],
    'elapsed_time': elapsed_time,
    'timestamp': timestamp,
    'parameters': {
        'N_CHAINS': N_CHAINS,
        'N_WARMUP': N_WARMUP,
        'N_SAMPLES_PER_CHAIN': N_SAMPLES_PER_CHAIN,
        'SWEEP_SIZE': SWEEP_SIZE,
        'N_ITER': N_ITER,
        'K': K,
    },
    'reference_energies': {
        'E_fcis_0': float(E_fcis[0]),
        'E_fcis_1': float(E_fcis[1]),
        'E_fcis_2': float(E_fcis[2]),
    }
}

with open(history_filepath, 'w', encoding='utf-8') as f:
    json.dump(history_to_save, f, indent=2, ensure_ascii=False)

logger.info(f"历史记录已保存到: {history_filepath}")

# 最终结果
logger.info("\n" + "="*60)
logger.info(f"训练完成!")
logger.info("="*60)

TypeError: cannot reshape array of shape (8,) (size 8) into shape (3, 4) (size 12)

$$
\begin{align*}
\Psi(\mathbf{x})^{-1}\hat{\mathcal{H}}\Psi(\mathbf{x})
&= \mathrm{Tr}\left[ \Psi^{-1}(\mathbf{x})\hat{H}\Psi(\mathbf{x}) \right]
\end{align*}
$$

In [ ]:
import matplotlib.pyplot as plt

# 创建 2行1列 的子图
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(12, 3))

# 第一个子图
ax1.plot(history['energy_0st'])
ax1.set_title('0st Energy')
ax1.set_ylabel('energy')

# 第二个子图
ax2.plot(history['loss'])
ax2.set_title('loss')
ax2.set_xlabel('step')
ax2.set_ylabel('energy')

# 第三个子图
ax3.plot(history['grad_norm'])
ax3.set_title('grad_norm')
ax3.set_xlabel('step')
ax3.set_ylabel('grad_norm')

plt.tight_layout()  # 自动调整间距
plt.show()

In [ ]:
N_CHAINS = 16
N_WARMUP = 100
N_SAMPLES_PER_CHAIN = 300
SWEEP_SIZE = 50
N_ITER =400
SINGLE_SIZE = hi.size  # 单个子系统维度 = 4

logger.info("\n" + "="*60)
logger.info("第二阶段训练开始 (带自然梯度)")
logger.info("="*60)
logger.info("训练参数配置:")
logger.info(f"  - N_CHAINS: {N_CHAINS}")
logger.info(f"  - N_WARMUP: {N_WARMUP}")
logger.info(f"  - N_SAMPLES_PER_CHAIN: {N_SAMPLES_PER_CHAIN}")
logger.info(f"  - SWEEP_SIZE: {SWEEP_SIZE}")
logger.info(f"  - N_ITER: {N_ITER}")

total_ansatz = NESTotalAnsatz(4,K,12,rngs=nnx.Rngs(11))
total_machine, total_graphdef,total_params = create_machine(total_ansatz)
total_matrix_machine, total_graphdef,total_params = create_machine_matrix(total_ansatz)

single_machine_list = []
for ansatz in total_ansatz.single_ansatz_list:
    m, g, p = create_single_machine(ansatz)
    single_machine_list.append(m)
    
    

optimizer = optax.sgd(learning_rate=0.01)
opt_state = optimizer.init(total_params)

ext_edges = []
for k in range(K):
    offset = k * SINGLE_SIZE
    for (i, j) in single_edges:
        ext_edges.append((i + offset, j + offset))
ext_edges = jnp.array(ext_edges)  # 转为jax数组（关键修复）

nes_rule = NESFermionHopRule(edges=ext_edges)
nes_sampler = nk.sampler.MetropolisSampler(
    hilbert=hi_ext,
    rule=nes_rule,
    n_chains=N_CHAINS,
    sweep_size=SWEEP_SIZE
)

# 采样器状态初始化（替代原 init_sampler_state）
sampler_rng = jax.random.PRNGKey(21)
sampler_state = nes_sampler.init_state(total_machine, total_params, sampler_rng)

# ==================== 训练循环（仅替换采样部分） ====================
logger.info("\n" + "="*60)
logger.info("开始多链 NES-VMC 训练 (NetKet 自定义采样器 + 自然梯度下降)")
logger.info("="*60)
logger.info(f"基态能量={E_fcis[0]:.8f} Ha| 第一激发态能量={E_fcis[1]:.8f} Ha| 第二激发态能量={E_fcis[2]:.8f} Ha")

history = {
    'step': [],
    'energy_0st': [],
    'energy_1st': [],
    'energy_std': [],
    'loss': [],
    'params': [],
    'E_Lmatrix':[],
    'natural_grad':[],
    'grad_flat':[],
    'samples':[],
    'log_Psi':[],
    'log_M':[],
    'log_Psi_mean':[],
    'log_Psi_min':[],
    'log_Psi_max':[],
    'grad_norm':[],
}

start_time = time.time()
for step in range(N_ITER):
    # 2. 正式采样
    samples_raw, sampler_state = nes_sampler.sample(
        machine=total_machine, parameters=total_params, 
        state=sampler_state, chain_length=N_SAMPLES_PER_CHAIN
    )
        # 3. 维度重塑，适配梯度函数输入
    samples = samples_raw.reshape(-1, hi_ext.size)
    x_batch = samples.reshape(-1, K, 4)
    # 3. 计算能量和自然梯度（逻辑和原代码一致）
    grad, loss_mean, E_L_mean = nes_vmc_gradient(ha=ha,
                                                 total_matrix_machine=total_matrix_machine,
                                                 total_machine=total_machine,
                                                 single_machine_list=single_machine_list,
                                                 total_params=total_params,
                                                 x_batch=samples.reshape(-1,K,4))
    #grad = jax.tree_util.tree_map(lambda x: x * 2, grad)
    
    grad_flat , grad_unravel_fn = ravel_pytree(grad)
    qgt_reg, unravel_fn = compute_qgt(total_machine, total_params, samples.reshape(-1,2,4), diag_shift=0.1)
    
    # # 自然梯度求解
    natural_grad_flat = jnp.linalg.solve(qgt_reg, grad_flat)
    natural_grad = grad_unravel_fn(natural_grad_flat)
    grad = natural_grad
        
    # 4. 更新参数
    updates, opt_state = optimizer.update(grad, opt_state, total_params)
    total_params = optax.apply_updates(total_params, updates)
    
    
    
    log_Psi_batch = total_machine(total_params, samples.reshape(-1,K,4))
    eig_vals, eig_vecs = jnp.linalg.eigh(E_L_mean)
    grad_norm = jnp.linalg.norm(grad_flat)
    
    
    history['step'].append(step)
    history['E_Lmatrix'].append(E_L_mean)
    history['samples'].append(samples)
    history['loss'].append(loss_mean)
    history['log_Psi_mean'].append(log_Psi_batch.mean())
    history['log_Psi_min'].append(log_Psi_batch.min())
    history['log_Psi_max'].append(log_Psi_batch.max())
    history['grad_norm'].append(grad_norm)
    history['energy_0st'].append(eig_vals[0])
    history['energy_1st'].append(eig_vals[1])
    history['params'].append(total_params)
    # 5. 记录历史
    if step % 50 == 0 or step == N_ITER - 1:
        # --------------------- 【NES-VMC 监控模板】直接用 ---------------------
        # 1. 监控 log_Psi
        #log_Psi_batch = total_machine(total_params, samples.reshape(-1,K,4))
        logger.info(f"log_Psi: mean={log_Psi_batch.mean():.3f} | min={log_Psi_batch.min():.3f} | max={log_Psi_batch.max():.3f}")

        # 2. 监控梯度范数
        
        logger.info(f"grad norm = {grad_norm:.4f}")

        # 5. 局域能量矩阵
        #logger.info(f"E_L mean =\n{E_L_mean}")
    
        #eig_vals, eig_vecs = jnp.linalg.eigh(E_L_mean)
        # #history['natural_grad'].append(natural_grad)
        # history['grad_flat'].append(grad_flat)
        # history['log_Psi'].append(log_Psi)
        # history['log_M'].append(log_M)
        
        logger.info(f"Step {step:3d} | Loss: {loss_mean}|0st能量={eig_vals[0]:.8f} Ha| 1st能量={eig_vals[1]:.8f} Ha")
        # logger.info(f'grad={grad_flat[30:31]}')
        logger.info('#-----------------------------------------#')


end_time = time.time()
elapsed_time = end_time - start_time
logger.info(f"训练耗时：{elapsed_time:.2f} 秒")

# 保存第二阶段历史记录
history_filename2 = f'history_{timestamp}_stage2.json'
history_filepath2 = os.path.join(log_dir, history_filename2)

history_to_save = {
    'step': history['step'],
    'energy_0st': [float(x) for x in history['energy_0st']],
    'energy_1st': [float(x) for x in history['energy_1st']],
    'loss': [float(x) for x in history['loss']],
    'log_Psi_mean': [float(x) for x in history['log_Psi_mean']],
    'log_Psi_min': [float(x) for x in history['log_Psi_min']],
    'log_Psi_max': [float(x) for x in history['log_Psi_max']],
    'grad_norm': [float(x) for x in history['grad_norm']],
    'elapsed_time': elapsed_time,
    'timestamp': timestamp,
    'stage': 'stage2_natural_gradient',
    'parameters': {
        'N_CHAINS': N_CHAINS,
        'N_WARMUP': N_WARMUP,
        'N_SAMPLES_PER_CHAIN': N_SAMPLES_PER_CHAIN,
        'SWEEP_SIZE': SWEEP_SIZE,
        'N_ITER': N_ITER,
        'K': K,
    },
    'reference_energies': {
        'E_fcis_0': float(E_fcis[0]),
        'E_fcis_1': float(E_fcis[1]),
        'E_fcis_2': float(E_fcis[2]),
    }
}

with open(history_filepath2, 'w', encoding='utf-8') as f:
    json.dump(history_to_save, f, indent=2, ensure_ascii=False)

logger.info(f"第二阶段历史记录已保存到: {history_filepath2}")

# 最终结果
logger.info("\n" + "="*60)
logger.info(f"训练完成!")
logger.info("="*60)

In [ ]:
import matplotlib.pyplot as plt


# 创建 2行1列 的子图
fig, (ax1, ax2, ax3, ax4) = plt.subplots(1, 4, figsize=(12, 3))
fig.suptitle("NES-VMC for H2 (K=2)", fontsize=14, fontweight='bold')

# 第一个子图
ax1.plot(history['energy_0st'])
ax1.hlines(y=E_fcis[0], xmin=0, xmax=400, color='r', linestyle='--', linewidth=2)
ax1.set_title('0st Energy')
ax1.set_ylabel('energy')
ax1.set_xlabel('step')


# 第一个子图
ax2.plot(history['energy_1st'])
ax2.hlines(y=E_fcis[1], xmin=0, xmax=400, color='r', linestyle='--', linewidth=2)
ax2.set_title('1st Energy')
ax2.set_ylabel('energy')
ax2.set_xlabel('step')



# 第二个子图
ax3.plot(history['loss'])
ax3.set_title('loss')
ax3.set_xlabel('step')
ax3.set_ylabel('energy')

# 第三个子图
ax4.plot(history['grad_norm'])
ax4.set_title('grad_norm')
ax4.set_xlabel('step')
ax4.set_ylabel('grad_norm')

plt.tight_layout()  # 自动调整间距
plt.show()

In [ ]:
def NES_loss_energy(ha, total_matrix_machine,single_machine_list,total_params, x):
    log_M = total_matrix_machine(total_params,x)
    Psi_Matrix = jnp.exp(log_M)
    # 添加正则化项，防止矩阵奇异
    #Psi_Matrix += 1e-6 * jnp.eye(Psi_Matrix.shape[0])
    H_psi_x = Ham_Psi(ha,single_machine_list,total_params,x)
    Psi_Matrix_inv = jnp.linalg.solve(Psi_Matrix, H_psi_x)
    return jnp.real(jnp.trace(Psi_Matrix_inv, axis1=-2, axis2=-1)), Psi_Matrix_inv